# Чекпоинт 7 — загрузка PRD-модели и тестовый предикт

Чистый блокнот: подключаемся к MLflow, грузим модель с алиасом **PRD** и делаем
предсказание на тестовых матчах.

## 1. Подключение

In [1]:
import os
import mlflow
import numpy as np
import pandas as pd
from prematch_data import load_prematch_data, RANDOM_STATE

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000')
MODEL_URI = 'models:/tennis-prematch-gb@PRD'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.get_tracking_uri()

'http://localhost:5000'

## 2. Загрузка модели по алиасу PRD

In [2]:
model = mlflow.sklearn.load_model(MODEL_URI)
model

D:\Projects\TennisPredictor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,loss,'log_loss'
,learning_rate,0.05
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,5
,min_impurity_decrease,0.0
,init,None


## 3. Тестовый предикт

In [3]:
data = load_prematch_data(seed=RANDOM_STATE)
X_sample = data.X_test[:10]
proba = model.predict_proba(X_sample)[:, 1]

out = data.meta_test.head(10)[['PLAYER_1_NAME', 'PLAYER_2_NAME', 'tourney_date', 'RESULT']].copy()
out['P(player1_win)'] = proba.round(3)
out['pred'] = (proba > 0.5).astype(int)
out['correct'] = (out['pred'] == out['RESULT']).astype(int)
out.reset_index(drop=True)

[prematch_data] loading cache from DATASETS/checkpoint7_cache


,PLAYER_1_NAME,PLAYER_2_NAME,tourney_date,RESULT,P(player1_win),pred,correct
0,Michael Chang,Jan Michael Gambill,2002-06-10,0,0.390,0,1
1,Pete Sampras,Alberto Martin,2001-08-06,0,0.751,1,0
2,Albert Portas,Emilio Benfele Alvarez,1998-04-13,0,0.591,1,0
3,Grigor Dimitrov,Filippo Volandri,2013-07-08,1,0.738,1,1
4,Alex De Minaur,Filip Krajinovic,2022-08-29,1,0.655,1,1
5,Cristiano Caratti,Michael Stich,1994-09-26,1,0.113,0,0
6,Mitchell Krueger,Luca Nardi,2024-07-29,1,0.548,1,1
7,Alex Calatrava,Jim Courier,1999-05-24,0,0.273,0,1
8,Simone Bolelli,Jaume Munar,2018-12-31,0,0.566,1,0
9,Hendrik Dreekmann,Stefan Edberg,1995-02-27,0,0.253,0,1


In [4]:
{'model_uri': MODEL_URI, 'sample_accuracy': round(float(out['correct'].mean()), 3)}

{'model_uri': 'models:/tennis-prematch-gb@PRD', 'sample_accuracy': 0.6}